In [1]:
from pyspark.sql import SparkSession

# Build the SparkSession
spark = SparkSession.builder \
    .appName("Jupyter PySpark Analytics") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.3.0


In [2]:
# Load Ratings
ratings_df = spark.read \
    .option("sep", "::") \
    .option("inferSchema", "true") \
    .csv("hdfs://namenode:9000/user/root/movielens/ratings.dat") \
    .toDF("UserID", "MovieID", "Rating", "Timestamp")

# Load Movies
movies_df = spark.read \
    .option("sep", "::") \
    .option("inferSchema", "true") \
    .csv("hdfs://namenode:9000/user/root/movielens/movies.dat") \
    .toDF("MovieID", "Title", "Genres")

# Show the first 5 rows to verify it worked!
movies_df.show(5, truncate=False)

+-------+----------------------------------+----------------------------+
|MovieID|Title                             |Genres                      |
+-------+----------------------------------+----------------------------+
|1      |Toy Story (1995)                  |Animation|Children's|Comedy |
|2      |Jumanji (1995)                    |Adventure|Children's|Fantasy|
|3      |Grumpier Old Men (1995)           |Comedy|Romance              |
|4      |Waiting to Exhale (1995)          |Comedy|Drama                |
|5      |Father of the Bride Part II (1995)|Comedy                      |
+-------+----------------------------------+----------------------------+
only showing top 5 rows



In [3]:
from pyspark.sql.functions import col, avg, count, desc, round

In [4]:
# Group by MovieID and count
popular_movies = ratings_df.groupBy("MovieID") \
    .agg(count("Rating").alias("Total_Ratings")) \
    .orderBy(desc("Total_Ratings"))

# Join with the movies_df to get the actual Titles instead of just IDs
popular_movies_with_titles = popular_movies.join(movies_df, "MovieID") \
    .select("Title", "Total_Ratings", "Genres") \
    .orderBy(desc("Total_Ratings"))

popular_movies_with_titles.show(10, truncate=False)

+-----------------------------------------------------+-------------+-----------------------------------+
|Title                                                |Total_Ratings|Genres                             |
+-----------------------------------------------------+-------------+-----------------------------------+
|American Beauty (1999)                               |2323         |Comedy|Drama                       |
|Star Wars: Episode IV - A New Hope (1977)            |1944         |Action|Adventure|Fantasy|Sci-Fi    |
|Star Wars: Episode V - The Empire Strikes Back (1980)|1928         |Action|Adventure|Drama|Sci-Fi|War  |
|Star Wars: Episode VI - Return of the Jedi (1983)    |1876         |Action|Adventure|Romance|Sci-Fi|War|
|Jurassic Park (1993)                                 |1830         |Action|Adventure|Sci-Fi            |
|Terminator 2: Judgment Day (1991)                    |1812         |Action|Sci-Fi|Thriller             |
|Saving Private Ryan (1998)                   

In [5]:
# Calculate average rating and total count per movie
movie_stats = ratings_df.groupBy("MovieID").agg(
    round(avg("Rating"), 2).alias("Average_Rating"),
    count("Rating").alias("Total_Ratings")
)

# Filter for movies with >= 1000 ratings, join with titles, and sort
top_rated_movies = movie_stats.filter(col("Total_Ratings") >= 1000) \
    .join(movies_df, "MovieID") \
    .select("Title", "Average_Rating", "Total_Ratings") \
    .orderBy(desc("Average_Rating"))

top_rated_movies.show(10, truncate=False)

+-----------------------------------------+--------------+-------------+
|Title                                    |Average_Rating|Total_Ratings|
+-----------------------------------------+--------------+-------------+
|Shawshank Redemption, The (1994)         |4.55          |1462         |
|Usual Suspects, The (1995)               |4.53          |1197         |
|Raiders of the Lost Ark (1981)           |4.52          |1615         |
|Godfather, The (1972)                    |4.52          |1410         |
|Schindler's List (1993)                  |4.51          |1544         |
|Star Wars: Episode IV - A New Hope (1977)|4.47          |1944         |
|Sixth Sense, The (1999)                  |4.42          |1530         |
|One Flew Over the Cuckoo's Nest (1975)   |4.39          |1102         |
|Casablanca (1942)                        |4.38          |1041         |
|Godfather: Part II, The (1974)           |4.38          |1090         |
+-----------------------------------------+--------

In [6]:
# Stop the Spark session
spark.stop()
print("Spark Session Stopped.")

Spark Session Stopped.
